# Kazakhstan Digital Knowledge Gaps
## Reproduction Notebook

This notebook reproduces the principal quantitative analyses reported in the study of digital knowledge gaps related to Kazakhstan across Kazakh Wikipedia, Wikidata, and large language models.

It uses the frozen datasets distributed with research release **v1.0**.

The notebook does not query live Wikimedia services or rerun AI model requests. This ensures that the reproduced results correspond to the fixed research snapshot used in the study.


## 1. Environment and data location

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the frozen research release.
# This works both during development and after the notebook
# is placed inside the published release directory.

candidates = [
    Path("."),
    Path("../research_release_v1.0"),
    Path("/content/drive/MyDrive/Kazakh_Wikipedia_Research/research_release_v1.0"),
]

DATA = None

for candidate in candidates:
    if (candidate / "kazakhstan_territorial_universe.csv").exists():
        DATA = candidate.resolve()
        break

if DATA is None:
    raise FileNotFoundError(
        "Could not locate research_release_v1.0 data files."
    )

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

print("Data directory:", DATA)


## 2. Kazakh Wikipedia corpus

This section reproduces the structural and article-depth summaries for the broader Kazakh Wikipedia corpus.


### Construction of the final analytical corpus

The initial namespace-0 analytical corpus contained **245,200 pages** after excluding redirects, pages identified as disambiguation pages in the Wikipedia dump, and the main page.

A second disambiguation check used Wikidata `P31`. An additional **789 pages** classified as `Q4167410` (*Wikimedia disambiguation page*) were excluded.

The resulting frozen analytical corpus therefore contains **244,411 pages**. This cleaned corpus is distributed directly with release v1.0 and is used for all subsequent analyses.

In [ ]:
# Load and verify the frozen final Kazakh Wikipedia corpus

corpus = pd.read_csv(
    DATA / "kkwiki_analytical_corpus.csv",
    low_memory=False
)

print("Final frozen analytical corpus:", f"{len(corpus):,}")

# The frozen file is already the final cleaned corpus.
# The earlier analytical corpus contained 245,200 pages.
# An additional 789 pages classified in Wikidata as
# Q4167410 (Wikimedia disambiguation page) were removed,
# resulting in 244,411 final analytical pages.

assert len(corpus) == 244411
assert corpus["page_id"].is_unique
assert (corpus["is_redirect"] == 0).all()
assert (corpus["is_disambiguation"] == 0).all()
assert (corpus["title"] != "Басты бет").all()

analysis_corpus = corpus.copy()

with_qid = analysis_corpus["qid"].notna().sum()
without_qid = analysis_corpus["qid"].isna().sum()

assert with_qid == 241506
assert without_qid == 2905

print(
    "With Wikidata QID:",
    f"{with_qid:,}",
    f"({with_qid / len(analysis_corpus) * 100:.2f}%)"
)

print(
    "Without Wikidata QID:",
    f"{without_qid:,}",
    f"({without_qid / len(analysis_corpus) * 100:.2f}%)"
)


In [ ]:
# Basic article-depth statistics

metrics = [
    "words",
    "references",
    "images",
    "internal_links",
    "categories",
]

depth_summary = (
    analysis_corpus[metrics]
    .agg(["count", "mean", "median"])
    .T
)

depth_summary["zero_n"] = [
    (analysis_corpus[m] == 0).sum()
    for m in metrics
]

depth_summary["zero_pct"] = (
    depth_summary["zero_n"]
    / len(analysis_corpus)
    * 100
)

depth_summary.round(2)


In [ ]:
# Link articles to Wikidata P31 classes and verified labels

p31 = pd.read_csv(
    DATA / "kkwiki_p31.csv",
    low_memory=False
)

labels = pd.read_csv(
    DATA / "p31_labels_verified.csv",
    low_memory=False
)

article_p31 = (
    analysis_corpus[
        analysis_corpus["qid"].notna()
    ]
    .merge(
        p31,
        on="qid",
        how="inner"
    )
    .merge(
        labels,
        on="p31",
        how="left"
    )
)

print("Article-P31 rows:", f"{len(article_p31):,}")
print("Distinct articles:", f"{article_p31['qid'].nunique():,}")
print("Distinct P31 classes:", f"{article_p31['p31'].nunique():,}")

labelled_classes = (
    article_p31[["p31", "label_en", "label_kk", "label_ru"]]
    .drop_duplicates("p31")
)

print(
    "P31 classes with at least one verified label:",
    f"{labelled_classes[['label_en','label_kk','label_ru']].notna().any(axis=1).sum():,}"
)


## 3. Kazakhstan-related Wikidata coverage

This section reproduces coverage and missing-article statistics for entities associated with Kazakhstan.


In [ ]:
# Reproduce overall Kazakhstan-related coverage

territorial = pd.read_csv(
    DATA / "kazakhstan_territorial_universe.csv",
    low_memory=False
)

assert territorial["qid"].is_unique

total = len(territorial)
covered = int(territorial["has_kkwiki"].sum())
missing = total - covered

coverage_table = pd.DataFrame({
    "status": ["covered", "missing", "total"],
    "entities": [covered, missing, total],
    "percent": [
        covered / total * 100,
        missing / total * 100,
        100.0
    ]
})

coverage_table["percent"] = coverage_table["percent"].round(2)

coverage_table


In [ ]:
# Check the principal coverage figures reported in the study

assert total == 29521
assert covered == 17866
assert missing == 11655

assert round(covered / total * 100, 2) == 60.52
assert round(missing / total * 100, 2) == 39.48

print("✓ Kazakhstan-related universe:", f"{total:,}")
print("✓ Covered:", f"{covered:,}", "(60.52%)")
print("✓ Missing:", f"{missing:,}", "(39.48%)")


In [ ]:
# Reproduce P31-level coverage from entity-level frozen data

p31_detail = pd.read_csv(
    DATA / "kazakhstan_p31_universe_final_detail.csv",
    low_memory=False
)

p31_labels = pd.read_csv(
    DATA / "p31_labels_verified.csv",
    low_memory=False
)

p31_reproduced = (
    p31_detail
    .groupby("p31", as_index=False)
    .agg(
        total_entities=("qid", "nunique"),
        covered_entities=(
            "qid",
            lambda x: x[
                p31_detail.loc[x.index, "has_kkwiki"].eq(1)
            ].nunique()
        ),
        missing_entities=(
            "qid",
            lambda x: x[
                p31_detail.loc[x.index, "has_kkwiki"].eq(0)
            ].nunique()
        ),
    )
)

p31_reproduced["coverage_pct"] = (
    p31_reproduced["covered_entities"]
    / p31_reproduced["total_entities"]
    * 100
).round(2)

p31_reproduced["missing_pct"] = (
    p31_reproduced["missing_entities"]
    / p31_reproduced["total_entities"]
    * 100
).round(2)

p31_reproduced = p31_reproduced.merge(
    p31_labels,
    on="p31",
    how="left"
)

print("P31 classes reproduced:", f"{len(p31_reproduced):,}")

p31_reproduced.sort_values(
    "total_entities",
    ascending=False
).head(10)


In [ ]:
# Verify reproduced P31 counts against the frozen released summary

released_p31 = pd.read_csv(
    DATA / "kazakhstan_p31_universe_final_summary.csv",
    low_memory=False
)

compare_cols = [
    "p31",
    "total_entities",
    "covered_entities",
    "missing_entities",
]

check = (
    released_p31[compare_cols]
    .merge(
        p31_reproduced[compare_cols],
        on="p31",
        how="outer",
        suffixes=("_released", "_reproduced"),
        indicator=True
    )
)

for col in [
    "total_entities",
    "covered_entities",
    "missing_entities",
]:
    check[f"{col}_match"] = (
        check[f"{col}_released"]
        == check[f"{col}_reproduced"]
    )

count_match = (
    (check["_merge"] == "both")
    & check[
        [
            "total_entities_match",
            "covered_entities_match",
            "missing_entities_match",
        ]
    ].all(axis=1)
)

print("Released P31 classes:", len(released_p31))
print("Reproduced P31 classes:", len(p31_reproduced))
print("Classes with matching counts:", int(count_match.sum()))

if count_match.all():
    print("✓ P31 class counts reproduced exactly.")
else:
    print("⚠ Some P31 class counts differ. Inspect before publication.")
    display(check.loc[~count_match].head(20))


## 4. Gender representation and article depth

This section reproduces the gender coverage and article-depth comparisons reported in the study.


### 4.1 Gender coverage

The authoritative gender coverage dataset for release v1.0 is
`kazakhstan_human_gender_corrected.csv`.

The corresponding frozen summary is
`kazakhstan_human_gender_corrected_summary.csv`.

The corrected dataset distinguishes four groups: Female, Male, Missing P21,
and Other / multiple.


In [ ]:
# GENDER COVERAGE — CORRECTED AUTHORITATIVE DATA

gender_corrected = pd.read_csv(
    DATA / "kazakhstan_human_gender_corrected.csv",
    low_memory=False
)

gender_summary_released = pd.read_csv(
    DATA / "kazakhstan_human_gender_corrected_summary.csv",
    low_memory=False
)

assert len(gender_corrected) == 6479
assert gender_corrected["qid"].nunique() == 6479

gender_summary_reproduced = (
    gender_corrected
    .groupby("gender", as_index=False)
    .agg(
        total=("qid", "size"),
        with_kkwiki=("has_kkwiki", "sum"),
    )
)

gender_summary_reproduced["without_kkwiki"] = (
    gender_summary_reproduced["total"]
    - gender_summary_reproduced["with_kkwiki"]
)

gender_summary_reproduced["coverage_pct"] = (
    gender_summary_reproduced["with_kkwiki"]
    / gender_summary_reproduced["total"]
    * 100
).round(2)

expected = (
    gender_summary_released
    .sort_values("gender")
    .reset_index(drop=True)
)

actual = (
    gender_summary_reproduced[
        expected.columns
    ]
    .sort_values("gender")
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    actual,
    expected,
    check_dtype=False,
)

print("✓ Corrected gender coverage reproduced exactly.")
display(actual)


### 4.2 Gender differences in article depth

Article-depth and inferential results are distributed as frozen research
outputs. They are retained unchanged because they correspond to the fixed
analytical snapshot used in the study.

The release includes the descriptive table together with continuous-metric
and categorical statistical tests.


In [ ]:
# Load frozen gender depth results

gender_depth = pd.read_csv(
    DATA / "kazakhstan_gender_depth.csv"
)

gender_continuous = pd.read_csv(
    DATA / "gender_depth_continuous_tests.csv"
)

gender_categorical = pd.read_csv(
    DATA / "gender_depth_categorical_tests.csv"
)

assert len(gender_depth) == 2
assert len(gender_continuous) == 5
assert len(gender_categorical) == 5

assert set(gender_depth["gender"]) == {"Female", "Male"}

female = gender_depth[
    gender_depth["gender"] == "Female"
].iloc[0]

male = gender_depth[
    gender_depth["gender"] == "Male"
].iloc[0]

assert int(female["articles"]) == 220
assert int(male["articles"]) == 962

print("✓ Frozen gender depth result tables verified.")
print()
print("Biography depth sample:")
print("Female:", int(female["articles"]))
print("Male:", int(male["articles"]))

print("\nContinuous/count tests:")
display(gender_continuous)

print("\nCategorical tests:")
display(gender_categorical)


## 5. AI benchmark

This section reproduces the reported recognition and factual benchmark statistics from the frozen scored datasets.

AI model requests are not rerun because model outputs may change over time.


### 5.1 LLM entity recognition benchmark

The benchmark compares matched entities with and without Kazakh Wikipedia
coverage.

The primary recognition endpoint is exact entity recognition
(`recognition_status == "correct_entity"`).

For each model, the covered and missing members of 126 matched entity pairs
are compared using the exact McNemar test. A 95% confidence interval for the
paired accuracy difference is estimated using 20,000 pair-bootstrap
resamples with seed `20260902`.

The model responses themselves are not regenerated; the frozen scored
response dataset is used.


In [ ]:
# AI RECOGNITION — PRIMARY REPRODUCTION

import numpy as np
import pandas as pd
from scipy.stats import binomtest

rec_ai = pd.read_csv(
    DATA / "ai_benchmark_recognition_scoring_FINAL_UNBLINDED_v1.1.csv",
    low_memory=False
)

assert len(rec_ai) == 504

rec_ai["correct_binary"] = (
    rec_ai["recognition_status"]
    .eq("correct_entity")
    .astype(int)
)

PAIR_COLS = ["p31", "pair_number"]

models = sorted(rec_ai["model"].unique())

# ------------------------------------------------------------
# Exact McNemar
# ------------------------------------------------------------

rec_rows = []

for model in models:

    d = rec_ai[
        rec_ai["model"] == model
    ].copy()

    wide = d.pivot(
        index=PAIR_COLS,
        columns="group",
        values="correct_binary"
    )

    assert len(wide) == 126
    assert wide.notna().all().all()

    covered_only = int(
        (
            (wide["covered"] == 1)
            &
            (wide["missing"] == 0)
        ).sum()
    )

    missing_only = int(
        (
            (wide["covered"] == 0)
            &
            (wide["missing"] == 1)
        ).sum()
    )

    discordant = covered_only + missing_only

    if discordant == 0:
        p = 1.0
    else:
        p = binomtest(
            min(covered_only, missing_only),
            n=discordant,
            p=0.5,
            alternative="two-sided"
        ).pvalue

    covered = wide["covered"].mean()
    missing = wide["missing"].mean()

    rec_rows.append({
        "model": model,
        "pairs": len(wide),
        "covered_correct": int(wide["covered"].sum()),
        "missing_correct": int(wide["missing"].sum()),
        "covered_pct": covered * 100,
        "missing_pct": missing * 100,
        "diff_pp": (covered - missing) * 100,
        "covered_only": covered_only,
        "missing_only": missing_only,
        "discordant_pairs": discordant,
        "mcnemar_exact_p": p,
    })

rec_reproduced = pd.DataFrame(rec_rows)


# ------------------------------------------------------------
# 20,000 paired bootstrap resamples
# Exact original RNG sequence retained
# ------------------------------------------------------------

rng = np.random.default_rng(20260902)
B = 20000

boot_rows = []

for model in models:

    d = rec_ai[
        rec_ai["model"] == model
    ].copy()

    wide = d.pivot(
        index=PAIR_COLS,
        columns="group",
        values="correct_binary"
    )

    pair_diff = (
        wide["covered"].to_numpy()
        -
        wide["missing"].to_numpy()
    )

    n = len(pair_diff)
    observed = pair_diff.mean()

    boot = np.empty(B)

    for i in range(B):
        idx = rng.integers(0, n, n)
        boot[i] = pair_diff[idx].mean()

    lo, hi = np.quantile(
        boot,
        [0.025, 0.975]
    )

    boot_rows.append({
        "model": model,
        "pairs": n,
        "difference_pp": observed * 100,
        "ci95_low_pp": lo * 100,
        "ci95_high_pp": hi * 100,
    })

rec_boot_reproduced = pd.DataFrame(boot_rows)


# ------------------------------------------------------------
# Verify against frozen result files
# ------------------------------------------------------------

rec_released = pd.read_csv(
    DATA / "recognition_primary_mcnemar_v1.1.csv"
)

boot_released = pd.read_csv(
    DATA / "recognition_primary_bootstrap_ci_v1.1.csv"
)

pd.testing.assert_frame_equal(
    rec_reproduced.sort_values("model").reset_index(drop=True),
    rec_released.sort_values("model").reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10,
)

pd.testing.assert_frame_equal(
    rec_boot_reproduced.sort_values("model").reset_index(drop=True),
    boot_released.sort_values("model").reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10,
)

print("✓ Recognition primary McNemar results reproduced exactly.")
print("✓ Recognition 20,000-resample bootstrap CIs reproduced exactly.")

display(rec_reproduced)
display(rec_boot_reproduced)


### 5.2 LLM factual benchmark

The factual benchmark contains 143 matched pair-property tests derived from
99 original matched entity pairs.

Some entity pairs contribute more than one factual property. Therefore the
143 pair-property observations are not treated as fully independent for
overall inference.

The overall accuracy difference is estimated descriptively across the 143
pair-property tests. Inferential uncertainty preserves the original
99-pair structure:

- 95% cluster-bootstrap CI: 100,000 resamples;
- two-sided cluster permutation test: 200,000 permutations;
- clustering unit: original matched entity pair;
- random seed: `20260902`.

All properties belonging to the same matched entity pair are resampled or
swapped together.


In [ ]:
# AI FACTUAL — PRIMARY CLUSTER-AWARE REPRODUCTION

fact_ai = pd.read_csv(
    DATA / "ai_benchmark_factual_scoring_FINAL_UNBLINDED_v1.0.csv",
    low_memory=False
)

assert len(fact_ai) == 572
assert fact_ai["prompt_id"].nunique() == 286

fact_ai["primary_correct"] = pd.to_numeric(
    fact_ai["primary_correct"],
    errors="raise"
).astype(int)

assert set(
    fact_ai["primary_correct"].unique()
) <= {0, 1}


# ------------------------------------------------------------
# Reconstruct matching keys independently
# ------------------------------------------------------------

fact_ai["pair_property_key_check"] = (
    fact_ai["p31"].astype(str)
    + "|"
    + fact_ai["pair_number"].astype(str)
    + "|"
    + fact_ai["property"].astype(str)
)

assert (
    fact_ai["pair_property_key_check"]
    ==
    fact_ai["pair_property_key"]
).all()

fact_ai["base_pair_key"] = (
    fact_ai["p31"].astype(str)
    + "|"
    + fact_ai["pair_number"].astype(str)
)

assert fact_ai["base_pair_key"].nunique() == 99
assert fact_ai["pair_property_key"].nunique() == 143


def make_factual_wide(sub):

    wide = (
        sub.pivot_table(
            index=[
                "base_pair_key",
                "pair_property_key",
                "p31",
                "p31_label",
                "pair_number",
                "property",
            ],
            columns="group",
            values="primary_correct",
            aggfunc="first",
        )
        .reset_index()
    )

    assert wide["covered"].notna().all()
    assert wide["missing"].notna().all()

    wide["covered"] = (
        wide["covered"].astype(int)
    )

    wide["missing"] = (
        wide["missing"].astype(int)
    )

    return wide


# ------------------------------------------------------------
# Overall descriptive result
# ------------------------------------------------------------

fact_desc_rows = []

for model in sorted(
    fact_ai["mapping_model"].unique()
):

    z = fact_ai[
        fact_ai["mapping_model"] == model
    ]

    wide = make_factual_wide(z)

    assert len(wide) == 143

    fact_desc_rows.append({
        "model": model,
        "pair_property_tests": len(wide),
        "covered_correct": int(
            wide["covered"].sum()
        ),
        "missing_correct": int(
            wide["missing"].sum()
        ),
        "covered_accuracy": (
            wide["covered"].mean()
        ),
        "missing_accuracy": (
            wide["missing"].mean()
        ),
        "difference": (
            wide["covered"].mean()
            -
            wide["missing"].mean()
        ),
        "covered_only": int(
            (
                (wide["covered"] == 1)
                &
                (wide["missing"] == 0)
            ).sum()
        ),
        "missing_only": int(
            (
                (wide["covered"] == 0)
                &
                (wide["missing"] == 1)
            ).sum()
        ),
    })

fact_desc_reproduced = pd.DataFrame(
    fact_desc_rows
)


# ------------------------------------------------------------
# Cluster bootstrap
# Original matched entity pair = cluster
# ------------------------------------------------------------

def cluster_bootstrap_overall(
    model_df,
    n_boot=100000,
    seed=20260902,
):

    wide = make_factual_wide(model_df)

    clusters = {
        k: z.copy()
        for k, z in wide.groupby(
            "base_pair_key"
        )
    }

    cluster_ids = list(
        clusters.keys()
    )

    assert len(cluster_ids) == 99

    observed = (
        wide["covered"].mean()
        -
        wide["missing"].mean()
    )

    rng = np.random.default_rng(seed)

    boot = np.empty(n_boot)

    for b in range(n_boot):

        sampled_ids = rng.choice(
            cluster_ids,
            size=len(cluster_ids),
            replace=True,
        )

        # Equivalent to concatenating the sampled
        # clusters, but avoids repeatedly creating
        # large temporary DataFrames.
        numerator = 0
        denominator = 0

        for k in sampled_ids:
            z = clusters[k]

            numerator += (
                z["covered"].sum()
                -
                z["missing"].sum()
            )

            denominator += len(z)

        boot[b] = (
            numerator / denominator
        )

    lo, hi = np.quantile(
        boot,
        [0.025, 0.975]
    )

    return observed, lo, hi


# ------------------------------------------------------------
# Cluster permutation test
# Swap condition labels for each original pair as a whole
# ------------------------------------------------------------

def cluster_permutation_overall(
    model_df,
    n_perm=200000,
    seed=20260902,
):

    wide = make_factual_wide(model_df)

    clusters = []

    for _, z in wide.groupby(
        "base_pair_key"
    ):

        d = (
            z["covered"].sum()
            -
            z["missing"].sum()
        )

        m = len(z)

        clusters.append((d, m))

    assert len(clusters) == 99

    total_tests = sum(
        m for _, m in clusters
    )

    d_arr = np.array(
        [d for d, _ in clusters],
        dtype=float,
    )

    observed = (
        d_arr.sum() / total_tests
    )

    rng = np.random.default_rng(seed)

    extreme = 0

    for i in range(n_perm):

        signs = rng.choice(
            [-1.0, 1.0],
            size=len(d_arr),
        )

        permuted = (
            (signs * d_arr).sum()
            /
            total_tests
        )

        if (
            abs(permuted)
            >= abs(observed)
        ):
            extreme += 1

    p = (
        1 + extreme
    ) / (
        n_perm + 1
    )

    return observed, p


# ------------------------------------------------------------
# Model-specific cluster inference
# ------------------------------------------------------------

fact_inf_rows = []

for model in sorted(
    fact_ai["mapping_model"].unique()
):

    z = fact_ai[
        fact_ai["mapping_model"] == model
    ]

    obs_b, lo, hi = (
        cluster_bootstrap_overall(
            z,
            n_boot=100000,
            seed=20260902,
        )
    )

    obs_p, perm_p = (
        cluster_permutation_overall(
            z,
            n_perm=200000,
            seed=20260902,
        )
    )

    assert abs(obs_b - obs_p) < 1e-12

    fact_inf_rows.append({
        "model": model,
        "base_pairs": 99,
        "pair_property_tests": 143,
        "difference": obs_b,
        "cluster_bootstrap_ci_low": lo,
        "cluster_bootstrap_ci_high": hi,
        "cluster_permutation_p": perm_p,
    })

fact_inf_reproduced = pd.DataFrame(
    fact_inf_rows
)


# ------------------------------------------------------------
# Verify against released final tables
# ------------------------------------------------------------

fact_desc_released = pd.read_csv(
    DATA /
    "factual_results_overall_descriptive_v1.0.csv"
)

fact_inf_released = pd.read_csv(
    DATA /
    "factual_results_overall_cluster_inference_v1.0.csv"
)

pd.testing.assert_frame_equal(
    fact_desc_reproduced
        .sort_values("model")
        .reset_index(drop=True),
    fact_desc_released
        .sort_values("model")
        .reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10,
)

pd.testing.assert_frame_equal(
    fact_inf_reproduced
        .sort_values("model")
        .reset_index(drop=True),
    fact_inf_released
        .sort_values("model")
        .reset_index(drop=True),
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-10,
)

print(
    "✓ Factual overall descriptive results reproduced exactly."
)

print(
    "✓ Factual 100,000-resample cluster-bootstrap CIs reproduced exactly."
)

print(
    "✓ Factual 200,000-permutation cluster tests reproduced exactly."
)

display(fact_desc_reproduced)
display(fact_inf_reproduced)


## 6. Verification against released master results

The final section compares reproduced results with the frozen master result tables included in release v1.0.
